In [1]:
import pandas as pd

In [2]:
ground_truth_df = pd.read_csv("../data/processed/validation/ground_truth.csv")
ground_truth_df.head(2)

,anchor_id,anchor_title,anchor_text,candidate_id,candidate_title,candidate_text,source,relevance_score
0,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",48299803,"Разработчик C#, Собственные продажи","Должность: Разработчик C#, Собственные продажи...",E5,2
1,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",49419678,"Разработчик JavaScript, Инструменты поисковой ...","Должность: Разработчик JavaScript, Инструменты...",TF-IDF,0


In [3]:
mask_e5 = ground_truth_df['source'].str.contains('E5', na=False, regex=False)
mask_tfidf = ground_truth_df['source'].str.contains('TF-IDF', na=False, regex=False)
mask_e5_tfidf = mask_e5 & mask_tfidf 
mask_random = ground_truth_df['source'].str.contains('Random', na=False, regex=False)

def print_score(name, mask):
    count = mask.sum()
    mean_score = ground_truth_df[mask]['relevance_score'].mean()
    left_part = f"Средний балл {count} кандидатов от {name}:"
    print(f"{left_part:<60} {mean_score:.2f} из 3.0")

print_score("E5", mask_e5)
print_score("TF-IDF", mask_tfidf)
print_score("E5 и TF-IDF (Пересечение)", mask_e5_tfidf)
print_score("Случайных (Шум)", mask_random)
print("-" * 75)


def calculate_hit_rate(mask):
    total = mask.sum()
    hits = len(ground_truth_df[mask & (ground_truth_df['relevance_score'] >= 2)])
    return (hits / total) * 100 if total > 0 else 0

def print_hit_rate(name, mask):
    hr = calculate_hit_rate(mask)
    left_part = f"Hit Rate {name}:"
    print(f"{left_part:<25} {hr:.1f}% релевантных")

print_hit_rate("E5", mask_e5)
print_hit_rate("TF-IDF", mask_tfidf)
print_hit_rate("E5 и TF-IDF", mask_e5_tfidf)
print_hit_rate("Random", mask_random)

Средний балл 984 кандидатов от E5:                           1.62 из 3.0
Средний балл 983 кандидатов от TF-IDF:                       1.62 из 3.0
Средний балл 204 кандидатов от E5 и TF-IDF (Пересечение):    2.11 из 3.0
Средний балл 500 кандидатов от Случайных (Шум):              0.06 из 3.0
---------------------------------------------------------------------------
Hit Rate E5:              50.0% релевантных
Hit Rate TF-IDF:          55.3% релевантных
Hit Rate E5 и TF-IDF:     71.6% релевантных
Hit Rate Random:          1.0% релевантных


In [4]:
ground_truth_df.shape

(2263, 8)

In [5]:
ground_truth_df['source'].value_counts()

source
E5           780
TF-IDF       779
Random       500
E5 TF-IDF    204
Name: count, dtype: int64